In [ ]:
# ===============================
# Data Wrangling & Feature Engineering
# ===============================

# Author: Shelly Resurreccion
# Purpose: This script performs data wrangling and feature engineering on the accident dataset to prepare it for predictive modeling.

---
### 1.0 Set Up

In [2]:
# ===============================
# Imports:
# ===============================
# --- Standard library ---
import sys
import os
import warnings

# --- Data handling ---
import pandas as pd
import numpy as np

# --- Preprocessing ---
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

# --- Data visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Warnings ---
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

In [3]:
# ===============================
# Functions:
# ===============================
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from scripts.data_i_o.write_csv import write_csv

---
### 2.0 Load Data and Review Values

In [19]:
# ===============================
# Load Data:
# ===============================
# Notes: The dataset was encoded in Latin-1 rather than UTF-8, so the encoding was explicitly specified when loading the CSV.
df = pd.read_csv(
    "../data/US_Accidents_March23.csv",
    encoding="latin1"
)

df.info()

df.columns.tolist()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246633 entries, 0 to 246632
Data columns (total 47 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Unnamed: 0             246633 non-null  int64  
 1   ID                     246633 non-null  object 
 2   Source                 246633 non-null  object 
 3   Severity               246633 non-null  int64  
 4   Start_Time             246633 non-null  object 
 5   End_Time               246633 non-null  object 
 6   Start_Lat              246633 non-null  float64
 7   Start_Lng              246633 non-null  float64
 8   End_Lat                246633 non-null  float64
 9   End_Lng                246633 non-null  float64
 10  Distance(mi)           246633 non-null  float64
 11  Description            246633 non-null  object 
 12  Street                 245863 non-null  object 
 13  City                   246631 non-null  object 
 14  County                 246633 non-nu

['Unnamed: 0',
 'ID',
 'Source',
 'Severity',
 'Start_Time',
 'End_Time',
 'Start_Lat',
 'Start_Lng',
 'End_Lat',
 'End_Lng',
 'Distance(mi)',
 'Description',
 'Street',
 'City',
 'County',
 'State',
 'Zipcode',
 'Country',
 'Timezone',
 'Airport_Code',
 'Weather_Timestamp',
 'Temperature_Range(F)',
 'Wind_Chill(F)',
 'Humidity(%)',
 'Pressure(in)',
 'Visibility(mi)',
 'Wind_Direction',
 'Wind_Speed(mph)',
 'Precipitation(in)',
 'Weather_Condition',
 'Amenity',
 'Bump',
 'Crossing',
 'Give_Way',
 'Junction',
 'No_Exit',
 'Railway',
 'Roundabout',
 'Station',
 'Stop',
 'Traffic_Calming',
 'Traffic_Signal',
 'Turning_Loop',
 'Sunrise_Sunset',
 'Civil_Twilight',
 'Nautical_Twilight',
 'Astronomical_Twilight']

In [5]:
# ===============================
# Identify numeric vs categorical
# ===============================
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df.select_dtypes(include=['object', 'category']).columns.tolist()

print("Numeric features:")
for col in numeric_features:
    print(f"  - {col}")

print("\nCategorical features:")
for col in categorical_features:
    print(f"  - {col}")

Numeric features:
  - Unnamed: 0
  - Severity
  - Start_Lat
  - Start_Lng
  - End_Lat
  - End_Lng
  - Distance(mi)
  - Wind_Chill(F)
  - Humidity(%)
  - Pressure(in)
  - Visibility(mi)
  - Wind_Speed(mph)
  - Precipitation(in)

Categorical features:
  - ID
  - Source
  - Start_Time
  - End_Time
  - Description
  - Street
  - City
  - County
  - State
  - Zipcode
  - Country
  - Timezone
  - Airport_Code
  - Weather_Timestamp
  - Temperature_Range(F)
  - Wind_Direction
  - Weather_Condition
  - Sunrise_Sunset
  - Civil_Twilight
  - Nautical_Twilight
  - Astronomical_Twilight


---
### 3.0 Feature Engineering - Match Part 1

In [6]:
# ===============================
# Timezone to UTC Offset
# ===============================
df = df[df["Timezone"].notna()].copy()

df["UTC_Offset_Hours"] = np.select(
    [
        df["Timezone"].str.lower().str.contains("eastern"),
        df["Timezone"].str.lower().str.contains("central"),
        df["Timezone"].str.lower().str.contains("mountain"),
        df["Timezone"].str.lower().str.contains("pacific"),
    ],
    [-5, -6, -7, -8],
    default=np.nan
)

In [7]:
# ===============================
# Temperature Range Processing
# ===============================
def parse_temp_range_f(temp_range):
    try:
        low, high = temp_range.split(" - ")
        return (float(low) + float(high)) / 2
    except:
        return np.nan

df["Temperature_F"] = df["Temperature_Range(F)"].apply(parse_temp_range_f)
df["Temperature_C"] = (df["Temperature_F"] - 32) * 5 / 9

df["Temperature_missing"] = df["Temperature_C"].isnull().astype(int)
df["Temperature_C"] = df["Temperature_C"].fillna(df["Temperature_C"].median())

In [8]:
# ===============================
# Median of Temperature in Celsius
# ===============================
median_temp_c = df["Temperature_C"].median()
median_temp_c

np.float64(9.444444444444445)

In [9]:
# Review new columns
df.head()[["UTC_Offset_Hours", "Temperature_Range(F)", "Temperature_F", "Temperature_C", "Temperature_missing"]]

,UTC_Offset_Hours,Temperature_Range(F),Temperature_F,Temperature_C,Temperature_missing
0,-6.0,31.0 - 35.0,33.0,0.555556,0
1,-5.0,30.0 - 34.0,32.0,0.000000,0
2,-5.0,29.0 - 33.0,31.0,-0.555556,0
3,-5.0,59.0 - 63.0,61.0,16.111111,0
4,-8.0,55.0 - 59.0,57.0,13.888889,0


---
### 4.0 Handling Missing Values

In [10]:
# ===============================
# Handle missing values (simple check)
# ===============================
missing_summary = df.isnull().sum().sort_values(ascending=False)
print("Missing values per column:\n", missing_summary)

Missing values per column:
 Precipitation(in)        10977
Wind_Chill(F)             8049
Wind_Direction            7554
Wind_Speed(mph)           7553
Visibility(mi)            6639
Humidity(%)               6518
Weather_Condition         6400
Temperature_F             5945
Temperature_Range(F)      5945
Pressure(in)              5437
Weather_Timestamp         5105
Astronomical_Twilight     1659
Nautical_Twilight         1659
Civil_Twilight            1659
Sunrise_Sunset            1659
Street                     770
Airport_Code               647
City                         2
Junction                     0
Station                      0
No_Exit                      0
Railway                      0
Roundabout                   0
Crossing                     0
Stop                         0
Traffic_Calming              0
Traffic_Signal               0
Turning_Loop                 0
UTC_Offset_Hours             0
Temperature_C                0
Give_Way                     0
Unnamed: 0 

In [11]:
# ===============================
## Review missing values in numerical columns
# ===============================
missing_cols = missing_summary[missing_summary > 0].index.tolist()
missing_review_numeric = []

for col in missing_cols:
    if df[col].dtype.kind in "biufc":  # numeric
        missing_review_numeric.append({
            "column": col,
            "missing_pct": df[col].isnull().mean(),
            "mean": df[col].mean(),
            "median": df[col].median(),
            "std": df[col].std(),
            "min": df[col].min(),
            "max": df[col].max()
        })

numeric_missing_df = pd.DataFrame(missing_review_numeric).sort_values(
    by="missing_pct", ascending=False
)

numeric_missing_df

,column,missing_pct,mean,median,std,min,max
0,Precipitation(in),0.044549,0.007272,0.00,0.035367,0.00,2.05
1,Wind_Chill(F),0.032666,45.298131,47.00,18.159411,-41.00,207.00
2,Wind_Speed(mph),0.030653,7.898008,7.00,5.823829,0.00,127.00
3,Visibility(mi),0.026944,8.443539,10.00,3.024719,0.00,70.00
4,Humidity(%),0.026452,68.523924,72.00,22.586025,1.00,100.00
5,Temperature_F,0.024127,47.950603,49.00,15.200429,-40.00,207.00
6,Pressure(in),0.022065,29.200444,29.58,1.342591,19.46,30.72


In [12]:
# ===============================
### Address numerical missingness
# ===============================
# Given low missingness and physical weather constraints, we used median imputation to preserve skewed distributions. 
# For variables where missingness could correlate with severe weather or sensor outages, we added explicit missing indicators to retain that signal.

# For "Precipitation(in)", create a missing indicator and fill missing values with median
df["Precipitation_missing"] = df["Precipitation(in)"].isnull().astype(int)
df["Precipitation(in)"] = df["Precipitation(in)"].fillna(df["Precipitation(in)"].median())

# For "Wind_Chill(F)", create a missing indicator and fill missing values with median
df["Wind_Chill_missing"] = df["Wind_Chill(F)"].isnull().astype(int)
df["Wind_Chill(F)"] = df["Wind_Chill(F)"].fillna(df["Wind_Chill(F)"].median())

# For "Wind_Speed(mph)", create a missing indicator and fill missing values with median
df["Wind_Speed_missing"] = df["Wind_Speed(mph)"].isnull().astype(int)
df["Wind_Speed(mph)"] = df["Wind_Speed(mph)"].fillna(df["Wind_Speed(mph)"].median())

# For "Visibility(mi)", create a missing indicator and fill missing values with median
df["Visibility_missing"] = df["Visibility(mi)"].isnull().astype(int)
df["Visibility(mi)"] = df["Visibility(mi)"].fillna(df["Visibility(mi)"].median())

# For "Humidity(%)", fill missing values with median - missing values are likely random
df["Humidity(%)"] = df["Humidity(%)"].fillna(df["Humidity(%)"].median())

# For "Pressure(in)", fill missing values with median - very tight distribution and median is close to the mean
df["Pressure(in)"] = df["Pressure(in)"].fillna(df["Pressure(in)"].median())

In [13]:
# ===============================
## Review missing values in categorical columns
# ===============================
missing_review_cat = []

for col in missing_cols:
    if df[col].dtype == "object" or df[col].dtype.name == "category":
        missing_review_cat.append({
            "column": col,
            "missing_pct": df[col].isnull().mean(),
            "n_unique": df[col].nunique(dropna=True),
            "top_values": df[col].value_counts(dropna=True).head(5).to_dict()
        })

categorical_missing_df = pd.DataFrame(missing_review_cat).sort_values(
    by="missing_pct", ascending=False
)

categorical_missing_df

,column,missing_pct,n_unique,top_values
0,Wind_Direction,0.030657,18,"{'CALM': 39716, 'W': 20036, 'S': 17573, 'N': 1..."
1,Weather_Condition,0.025974,80,"{'Fair': 97204, 'Cloudy': 47653, 'Mostly Cloud..."
2,Temperature_Range(F),0.024127,120,"{'48.0 - 52.0': 8274, '52.0 - 56.0': 7448, '46..."
3,Weather_Timestamp,0.020718,26004,"{'2023-01-31 07:53:00': 331, '2023-01-19 16:53..."
4,Astronomical_Twilight,0.006733,2,"{'Day': 187708, 'Night': 57037}"
5,Nautical_Twilight,0.006733,2,"{'Day': 175899, 'Night': 68846}"
6,Civil_Twilight,0.006733,2,"{'Day': 160868, 'Night': 83877}"
7,Sunrise_Sunset,0.006733,2,"{'Day': 146933, 'Night': 97812}"
8,Street,0.003125,45560,"{'I-5 N': 2532, 'I-5 S': 2205, 'I-95 N': 1960,..."
9,Airport_Code,0.002626,1529,"{'KCQT': 4557, 'KEMT': 2754, 'KRDU': 2612, 'KF..."


In [14]:
# ===============================
### Address categorical missingness
# ===============================
# For categorical features, we avoid mode imputation when missingness may represent a distinct state. Instead, we introduce explicit ‘Unknown’ 
# categories, especially for high-cardinality or sensor-driven variables. Deterministic binary features with minimal missingness are safely mode-imputed.

# For "Wind_Direction", fill missing values with "Unknown". Missing does not mean "Calm" wind conditions as this is a separate category.
df["Wind_Direction"] = df["Wind_Direction"].fillna("Unknown")

# For "Weather_Condition", fill missing values with "Unknown". Missing does not mean "Fair" conditions as this is a separate category.
df["Weather_Condition"] = df["Weather_Condition"].fillna("Unknown")

# For "Temperature_Range(F)", convert to Celisius (as done in Part 1) and fill missing values with median after creating missing indicator. Completed above!

# For "Weather_Timestamp", do not impute. Missingness could be likely due to sensor issues. I will rely on start and end time instead. 
df["Start_Time_clean"] = df["Start_Time"].str.strip().str[:19]
dt = pd.to_datetime(df["Start_Time_clean"], errors="coerce", infer_datetime_format=True)

df["Start_Hour"] = dt.dt.hour
df["Start_Dayofweek"] = dt.dt.dayofweek
df["Start_month"] = dt.dt.month

# Create "Day_Night" feature based on "weather_hour", use this to determine if it's day or night to fill in Nautical_Twilight, Civil_Twilight, Sunrise_Sunset and Astronomical_Twilight
def is_day(hour):
    if pd.isna(hour):
        return np.nan
    return "Day" if 6 <= hour < 18 else "Night"

df["Day_Night"] = df["Start_Hour"].apply(is_day)

# For "Airport_Code", fill missing values with "Unknown". Missing may indicate no nearby airport.
df["Airport_Code"] = df["Airport_Code"].fillna("Unknown")

# For "Street", fill missing values with "Unknown". This is extremely high cardinality and missing values are likely random. Likely to drop or use other location features instead.
df["Street"] = df["Street"].fillna("Unknown")

# For "Timezone", fill missing values with mode. Missing values are likely random.
df["Timezone"] = df["Timezone"].fillna(df["Timezone"].mode()[0])

# For "Zipcode", imputation is not needed but filling missing values with "Unknown" if model requires it. 
df["Zipcode"] = df["Zipcode"].fillna("Unknown")

# For "City", imputation is not needed but filling missing values with "Unknown" if model requires it.
df["City"] = df["City"].fillna("Unknown")

/var/folders/39/m2n3qgr96q700lmqw2r65zbh0000gn/T/ipykernel_1424/3581830072.py:17: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  dt = pd.to_datetime(df["Start_Time_clean"], errors="coerce", infer_datetime_format=True)


In [15]:
# Drop columns with excessive missingness or not useful for modeling
df = df.drop(columns=[
    "Unnamed: 0",
    "Temperature_F",
    "Temperature_Range(F)",
    "Weather_Timestamp",
    "Nautical_Twilight",
    "Civil_Twilight",
    "Sunrise_Sunset",
    "Astronomical_Twilight"
])

In [16]:
# ===============================
## Review missing values again after imputation
# ===============================
missing_summary = df.isnull().sum().sort_values(ascending=False)
print("Missing values per column:\n", missing_summary)

# Re-identify numeric vs categorical after imputation
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df.select_dtypes(include=['object', 'category']).columns.tolist()

Missing values per column:
 ID                       0
Source                   0
Bump                     0
Crossing                 0
Give_Way                 0
Junction                 0
No_Exit                  0
Railway                  0
Roundabout               0
Station                  0
Stop                     0
Traffic_Calming          0
Traffic_Signal           0
Turning_Loop             0
UTC_Offset_Hours         0
Temperature_C            0
Temperature_missing      0
Precipitation_missing    0
Wind_Chill_missing       0
Wind_Speed_missing       0
Visibility_missing       0
Start_Time_clean         0
Start_Hour               0
Start_Dayofweek          0
Start_month              0
Amenity                  0
Weather_Condition        0
Precipitation(in)        0
City                     0
Severity                 0
Start_Time               0
End_Time                 0
Start_Lat                0
Start_Lng                0
End_Lat                  0
End_Lng                  0


---
### 5.0 Export Cleaned Data

In [18]:
# ===============================
## Export Data 
# ===============================
write_csv(df, "../data/transformed_data.csv")

DataFrame written to CSV file
